In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

from dotenv import load_dotenv
import os

In [34]:
# load api key
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [35]:
# load models 
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8441.22it/s]


In [36]:
# load the pdf
pdf_link = "data/edital-mestrado.pdf"
loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()

In [37]:
# separate the chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)

In [38]:
# save the chunks in the vectordb
vectordb = Chroma(embedding_function=embedding_model, persist_directory="rerankDB")
vectordb.add_documents(chunks)

['ce97c8e0-1cee-4706-b893-7938ef0c7bf1',
 'e437bd5e-1ff2-4fd0-90b9-6b68afddc83e',
 '0d200a68-35a4-4d4c-9f28-9b14ae7e4dc2',
 '7a04210c-0384-4fe7-9e51-7e21c27ea9c5',
 '6ddf1119-8a02-4961-b7c4-0db568940e8c',
 'ab22b274-8bee-423c-a873-312ad3291c89',
 '7ce5fc7c-7075-47d0-beae-b7abba8cc05d',
 'b1c0a236-3e3b-4844-b01b-3508a2e30569',
 '6b81c854-8b28-4147-9ad4-d4acd4609aaf',
 'dc85dd96-1942-4202-baeb-0fe5525222bf',
 'cb77f436-b968-4008-bc71-2692bef62a82',
 '5f6dd453-454c-40b3-a43f-236c04401b58',
 '0c26c742-ed97-4691-99fe-600fd13e4ce1',
 '394d170f-58d7-4094-9436-8313352c2416',
 '02e4ce20-a89a-4ea9-8361-6f2068e19124']

In [ ]:
# load the db, k = 10 to send more options for reranking and compression
naive_retriever = vectordb.as_retriever(search_kwargs={"k":10})

In [40]:
model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3"
)

rerank = CrossEncoderReranker(
    model=model,
    top_n=3
)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 12392.39it/s]


In [41]:
compressor_retriever = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=naive_retriever
)

In [42]:
TEMPLATE = """
    Você é um secretário da coordenação de pós-graduação da UFCG. Responda a pergunta abaixo utilizando o contexto informado.
    Query:
    {question}

    Context:
    {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [43]:
setup_retrieval = RunnableParallel({"question": RunnablePassthrough(), "context": compressor_retriever})
output_parser = StrOutputParser()
compressor_retrieval_chain = setup_retrieval |rag_prompt | llm | output_parser

In [46]:
query = "Quando vai ser o período de matrícula?"

docs = compressor_retriever.invoke(query)

print(f"{len(docs)} documentos enviados ao LLM\n")

for i, doc in enumerate(docs, 1):
    print("=" * 80)
    print(f"Documento {i}")
    print(doc.metadata)
    print(doc.page_content)

response = compressor_retrieval_chain.invoke(query)

print("\n" + "=" * 80)
print("RESPOSTA DO LLM")
print("=" * 80)
print(response)

3 documentos enviados ao LLM

Documento 1
{'total_pages': 12, 'title': 'Sistema Único de Processo Eletrônico em Rede - 23096.038647/2026-35', 'source': 'data/edital-mestrado.pdf', 'page': 3, 'creationdate': '2026-06-08T12:18:58+00:00', 'moddate': '2026-06-08T12:18:58+00:00', 'page_label': '4', 'producer': 'Skia/PDF m148', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36', 'start_index': 0}
A pontuação ﬁnal será contabilizada conforme a ﬁcha de avaliação (Anexo IV A/B), sendo os valores máximos de 162,40 (cento e sessenta e dois vírgulaquarenta) para o mestrado e 224,70 (duzentos e vinte e quatro vírgula setenta) para o doutorado.
6.  DA APROVAÇÃO E CLASSIFICAÇÃO
Para os candidatos de Mestrado, serão considerados classiﬁcados os(as) candidatos(as) que atingirem a pontuação ﬁnal igual ou superior a 72,00 (setenta edois vírgula zero). Para os candidatos de Doutorado, serão considerados(as) classiﬁcados(as) os(as) c